# Notebook 04 — Model Building

**Maternal & Infant Mortality Risk Prediction Using Data Mining Techniques**

> ⚠️ **SYNTHETIC — generated for academic demonstration. Not real clinical data.**

---

## Models Trained

| Category | Algorithm | Why? |
|----------|-----------|------|
| Classification | Decision Tree | Interpretable; mirrors clinical decision logic |
| Classification | Random Forest | Ensemble bagging reduces variance |
| Classification | Naïve Bayes | Fast probabilistic baseline |
| Classification | Logistic Regression | Linear baseline with probabilities |
| Classification | KNN | Non-parametric local patterns |
| Classification | SVM | Margin maximisation for generalisation |
| Classification | XGBoost | State-of-the-art gradient boosting |
| Clustering | K-Means | Unsupervised structure discovery |
| Association | Apriori (mlxtend) | Rule mining for interpretable patterns |

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing import preprocess_data, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.models import (
    train_all_classifiers,
    run_kmeans_analysis,
    run_apriori,
)

FIGURES = os.path.join('..', 'reports', 'figures')
os.makedirs(FIGURES, exist_ok=True)

print('Imports OK')

## 1. Load & Preprocess Data

In [ ]:
# Load raw synthetic data
df = pd.read_csv(os.path.join('..', 'data', 'raw', 'synthetic_maternal_infant_risk.csv'))

# Preprocess: split, encode, SMOTE on training only
X_train, X_test, y_train, y_test, preprocessor, feature_names = preprocess_data(
    df, test_size=0.2, random_state=42, apply_smote=True
)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Features: {len(feature_names)}")

## 2. Classification — Train All Models with GridSearchCV

In [ ]:
# Train 7 classifiers with 5-fold GridSearchCV, optimising for RECALL
# because a false negative (missing a HIGH-RISK pregnancy) is clinically dangerous.

best_models = train_all_classifiers(X_train, y_train, cv=5, scoring='recall')

print("\n✅ All models trained.")
for name, model in best_models.items():
    print(f"  {name}: {type(model).__name__}")

## 3. Clustering — K-Means (Elbow + Silhouette)

In [ ]:
# K-Means clustering — useful for discovering natural groupings that may
# correspond to risk strata (e.g., healthy vs borderline vs critical).

print("Running K-Means for k=2..10:")
kmeans_results, best_km = run_kmeans_analysis(X_train, k_range=range(2, 11))

# Convert to DataFrame for plotting
km_df = pd.DataFrame(kmeans_results)

In [ ]:
# Elbow & Silhouette plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Elbow
axes[0].plot(km_df['k'], km_df['inertia'], 'bo-', linewidth=2)
axes[0].set_title('SYNTHETIC — K-Means Elbow Method', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(km_df['k'], km_df['silhouette'], 'rs-', linewidth=2)
axes[1].set_title('SYNTHETIC — Silhouette Score vs k', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

# Mark best k
best_k = km_df.loc[km_df['silhouette'].idxmax(), 'k']
axes[1].axvline(x=best_k, color='green', linestyle='--', label=f'Best k={best_k}')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig10_kmeans_elbow_silhouette.png'), dpi=150)
plt.show()
print(f'Best k by silhouette: {best_k}')
print('Saved: fig10_kmeans_elbow_silhouette.png')

## 4. Association Rules — Apriori

In [ ]:
# Apriori association rule mining on binned attributes.
# We bin continuous variables into clinically meaningful categories
# (e.g., anemia severity, BP category) and mine for rules that co-occur
# with HIGH-RISK outcomes.

rules = run_apriori(df, min_support=0.03, min_confidence=0.25, top_n=10)

if len(rules) > 0:
    # Display top 10 rules by lift
    display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
    print("\nSYNTHETIC — Top 10 Association Rules by Lift:")
    print(rules[display_cols].to_string())
else:
    print("No rules found. Try adjusting min_support / min_confidence.")

## 5. Save Models for Evaluation (Notebook 05)

We pass the trained models forward to the evaluation notebook via joblib.

In [ ]:
import joblib

# Save all trained models temporarily for the evaluation notebook
models_cache = os.path.join('..', 'models', 'all_trained_models.joblib')
joblib.dump({
    'models': best_models,
    'preprocessor': preprocessor,
    'feature_names': feature_names,
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
}, models_cache)

print(f"All trained models + data saved → {os.path.abspath(models_cache)}")

## Summary

- **7 classifiers** trained with GridSearchCV (5-fold), optimised for RECALL.
- **K-Means** clustering analysed with elbow and silhouette methods.
- **Apriori** association rules mined on binned clinical attributes.
- All models cached to `models/` for evaluation in Notebook 05.

**Next → Notebook 05: Evaluation**